In [1]:
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()

In [2]:
# get the channel slug, episode guid, and transcription uuid for all transcriptions without a spaCy segmentation
transcripts = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            # check if any of the objects in segmentation_set have name == "spaCy"
            if podcast["language"].startswith("en") and not any(segmentation['name'] == "spaCy" for segmentation in transcription['segmentation_set']):
                transcripts.append({"slug": podcast['slug'], "ep_guid": audioitem['guid'], "trans_uuid": transcription['uuid']})
len(transcripts)

12

In [4]:
import sentence_splitter
import spacy

for transcript in transcripts:
    slug = transcript["slug"]
    ep_guid = transcript["ep_guid"]
    trans_uuid = transcript["trans_uuid"]

    # Get the text from the API
    data = requests.get(f"{SERVER}:{PORT}/api/transcriptions/{trans_uuid}/")
    transcript = data.json()

    # Split the text into sentences
    utterances = sentence_splitter.sentence_splitter(transcript, "en_core_web_lg")

    segmentation_dict = {
        "uuid": trans_uuid,
        "name": "spaCy",
        "segmentor": {"name": "spaCy", "version": spacy.__version__},
        "utterance_set": [utterance for utterance in utterances if utterance.get("text") != ""]
    }

    res = requests.post(f"{SERVER}:{PORT}/api/podcasts/{slug}/{ep_guid}/utterances/", json=segmentation_dict)
    print(res.status_code)

UnboundLocalError: local variable 'word' referenced before assignment

In [5]:
# copy a segmentation and write it back to the transcription with a new name

import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()

target_segmentations = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    target_segmentations.append(segmentation)
target_segmentations


[{'transcription': 1,
  'uuid': '7909535e-dad4-11ed-ba56-00155d8020a1',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 2,
  'uuid': '7ad3aa86-dad4-11ed-980f-00155d8020a1',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 3,
  'uuid': '7cac1e2e-dad4-11ed-9e52-00155d8020a1',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 4,
  'uuid': '7e2630e6-dad4-11ed-9c3e-00155d8020a1',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 5,
  'uuid': '9a44cada-dad4-11ed-9644-00155d8020a1',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 106,
  'uuid': '8303cd26-dca5-11ed-bbbf-00155d0d192b',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 107,
  'uuid': '1fecf512-dca7-11ed-878d-00155d0d192b',
  'name': 'spaCy',
  'segmentor': {'name': 'spaC

## FILTER TO NEW SEGMENTATION

In [3]:
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()

In [5]:
import numpy as np
import pandas as pd
import spacy
import requests

for podcast in podcasts[1:]:
    slug = podcast['slug']
    for audioitem in podcast['audioitem_set']:
        ep_guid = audioitem['guid']
        for transcription in audioitem['transcription_set']:
            trans_uuid = transcription['uuid']
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()

                    df = pd.DataFrame(seg["utterance_set"])
                    df["score"] = None

                    # get the "ClaimBuster-BBA-(COREF)" score for each utterance where it exists, otherwise use the "ClaimBuster-BBA" score
                    def get_score(classification_set):
                        score = next((float(cl["label"]) for cl in classification_set if cl["agent"] == "ClaimBuster-BBA-(COREF)"), None)
                        if score is None:
                            score = next((float(cl["label"]) for cl in classification_set if cl["agent"] == "ClaimBuster-BBA"), None)
                            if score is None:
                                score = 0
                        return score
                    
                    df["score"] = df["classification_set"].apply(get_score)
                    # set all records that score below the 95th percentile to hidden
                    df.loc[df["score"] < df["score"].quantile(0.95), "visibility"] = 0
                    new_utterances = df.drop(columns=["score"]).to_dict(orient="records")


                    segmentation_dict = {
                        "uuid": trans_uuid,
                        "name": "spaCy-5% most CW",
                        "segmentor": {"name": "spaCy", "version": spacy.__version__},
                        "utterance_set": new_utterances
                    }
                    res = requests.post(f"{SERVER}:{PORT}/api/podcasts/{slug}/{ep_guid}/utterances/", json=segmentation_dict)
                    print(res.status_code)


201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201


In [ ]:
res.text

'<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta http-equiv="content-type" content="text/html; charset=utf-8">\n  <meta name="robots" content="NONE,NOARCHIVE">\n  <title>TypeError\n          at /api/podcasts/verdict-with-ted-cruz/bec2ea7c-5985-449f-bec2-afd5002d192b/utterances/</title>\n  <style type="text/css">\n    html * { padding:0; margin:0; }\n    body * { padding:10px 20px; }\n    body * * { padding:0; }\n    body { font:small sans-serif; background-color:#fff; color:#000; }\n    body>div { border-bottom:1px solid #ddd; }\n    h1 { font-weight:normal; }\n    h2 { margin-bottom:.8em; }\n    h3 { margin:1em 0 .5em 0; }\n    h4 { margin:0 0 .5em 0; font-weight: normal; }\n    code, pre { font-size: 100%; white-space: pre-wrap; word-break: break-word; }\n    summary { cursor: pointer; }\n    table { border:1px solid #ccc; border-collapse: collapse; width:100%; background:white; }\n    tbody td, tbody th { vertical-align:top; padding:2px 3px; }\n    thead th {\n      padding:1px 6

In [ ]:
seg_uuid = target_segmentations[0]['uuid']
seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
seg = seg.json()

In [ ]:
# find the 5% of utterances with the highest average Checkworthiness score from both CB & FV
import numpy as np
import pandas as pd
df = pd.DataFrame(seg["utterance_set"])

# get the average Checkworthiness score for each utterance, from the label field of the classification objects
df["score"] = df["classification_set"].apply(lambda x: np.mean([float(cl["label"]) for cl in x if cl["category"] == "Checkworthy" and (cl["agent"] == "ClaimBuster-BBA" or cl["agent"] == "Factiverse")]))


In [ ]:
# set all records that score below the 95th percentile to hidden
df.loc[df["score"] < df["score"].quantile(0.95), "hidden"] = True

In [ ]:
utterance_set = df.drop(columns=["score"]).to_dict(orient="records")

[{'hidden': True,
  'start': '0.92',
  'end': '3.74',
  'speaker': 'SPEAKER_09',
  'text': 'Welcome, it is a verdict with Senator Ted Cruz.',
  'text_coref': None,
  'microfacts': None,
  'claimspan': None,
  'uuid': '7909b650-dad4-11ed-ba56-00155d8020a1',
  'classification_set': [{'utterance': 427391,
    'qualifier': 'Checkworthiness',
    'label': '0.15125917749913328',
    'category': 'Checkworthy',
    'agent': 'ClaimBuster-BBA'},
   {'utterance': 427391,
    'qualifier': 'Checkworthiness',
    'label': '0',
    'category': 'Checkworthy',
    'agent': 'Factiverse'}],
  'query_set': []},
 {'hidden': True,
  'start': '3.86',
  'end': '5.06',
  'speaker': 'SPEAKER_09',
  'text': 'Ben Ferguson with you.',
  'text_coref': None,
  'microfacts': None,
  'claimspan': None,
  'uuid': '790a089e-dad4-11ed-ba56-00155d8020a1',
  'classification_set': [{'utterance': 427392,
    'qualifier': 'Checkworthiness',
    'label': '0.11236805868623494',
    'category': 'Checkworthy',
    'agent': 'Claim